In [13]:
import pandas as pd

In [14]:
df = pd.read_csv('../data/processed/real_estate_listings_zain.csv')

In [15]:
df.head()

,id,url,type,purpose,area,bedroom,bath,added,price,location,location_city,source
0,1,https://www.zameen.com/Property/askari_askari_...,Flat,For Sale,10 Marla,3,3.0,33 minutes ago,PKR 3 Crore,"Askari 11, Askari",Lahore,Zameen
1,2,https://www.zameen.com/Property/gulberg_3_gulb...,Other,For Sale,2.4 Kanal,6,7.0,1 hour ago,PKR 15.5 Crore,"Gulberg 3 - Block M, Gulberg 3",Lahore,Zameen
2,3,https://www.zameen.com/Property/dha_phase_5_pe...,Apartment,For Sale,9.8 Marla,3,4.0,3 hours ago,PKR 6.95 Crore,"Penta Square By DHA Lahore, DHA Phase 5",Lahore,Zameen
3,4,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5,7.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block Y, DHA Phase 7",Lahore,Zameen
4,5,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5,6.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block U, DHA Phase 7",Lahore,Zameen


In [16]:
df[df['purpose'] != 'For Sale']

,id,url,type,purpose,area,bedroom,bath,added,price,location,location_city,source
509,510,https://www.zameen.com/Property/dha_phase_7_se...,Room,For Rent,106 Sq. Yd.,2,2.0,1 hour ago,PKR 1.7 Crore,"Sehar Commercial Area, DHA Phase 7",Karachi,Zameen
795,796,https://www.zameen.com/Property/e_11_e-11_4_e-...,Room,For Rent,4 Marla,1,1.0,1 hour ago,PKR 1.18 Crore,"E-11/4, E-11",Islamabad,Zameen
2236,2237,https://www.zameen.com/Property/dha_phase_6_dh...,House,For Rent,1 Kanal,5,6.0,4 hours ago,PKR 3.25 Lakh,"DHA Phase 6 - Block K, DHA Phase 6",Lahore,Zameen
2237,2238,https://www.zameen.com/Property/dha_defence_dh...,House,For Rent,5 Marla,3,3.0,5 hours ago,PKR 1.15 Lakh,"DHA 9 Town, DHA Defence",Lahore,Zameen
2238,2239,https://www.zameen.com/Property/dha_defence_dh...,House,For Rent,10 Marla,4,5.0,12 hours ago,PKR 2.7 Lakh,"DHA Phase 5, DHA Defence",Lahore,Zameen
...,...,...,...,...,...,...,...,...,...,...,...,...
4481,4482,https://www.zameen.com/Property/multan_buch_ex...,House,For Rent,6.5 Marla,5,5.0,1 month ago,PKR 1.15 Lakh,"Buch Executive Villas, Multan",Multan,Zameen
4482,4483,https://www.zameen.com/Property/multan_cantt_8...,Farm House,For Rent,4 Kanal,8,6.0,1 month ago,PKR 3.5 Lakh,"Cantt, Multan",Multan,Zameen
4483,4484,https://www.zameen.com/Property/wapda_town_wap...,House,For Rent,10 Marla,5,5.0,1 month ago,PKR 1.5 Lakh,"Wapda Town Phase 2, Wapda Town",Multan,Zameen
4484,4485,https://www.zameen.com/Property/multan_bahadur...,House,For Rent,7 Marla,4,4.0,2 months ago,PKR 65 Thousand,"Bahadurpur, Multan",Multan,Zameen


In [17]:
rent_houses = df[df['purpose'] != 'For Sale']
rent_houses.head()
rent_houses.shape
rent_houses.to_csv('../data/processed/rent_houses.csv', index=False)


In [18]:
df['purpose'].value_counts(dropna=False)

purpose
For Sale    3108
For Rent    2252
Name: count, dtype: int64

In [19]:
df['type'].value_counts(dropna=False)

type
House               2654
Other                667
Room                 477
Apartment            447
Flat                 319
Upper Portion        216
Plot                 164
Villa                129
Residential Plot     116
Lower Portion         54
Shop                  24
Farm House            22
Commercial Plot       22
Office                17
Building              13
Penthouse             12
File                   5
Factory                1
Land                   1
Name: count, dtype: int64

In [20]:
df_sale = df[df['purpose'] == 'For Sale'].copy()
df_sale.to_csv('../data/processed/sale_houses_zain.csv', index=False)
# from here on, work only with df_sale

In [21]:
df_sale.head()
df_sale.shape


(3108, 12)

## Explore price and area units

Before converting anything, see exactly which unit words show up in `price` and `area` -- that's what price_pkr / area_sqft need to cover.

In [22]:
import re

# unit word = whatever's left after stripping the currency prefix (PKR / Rs) and the number
def price_unit(s):
    m = re.match(r'^(?:PKR|Rs\.?)\s*[\d,\.]+\s*(.*)$', str(s).strip(), re.IGNORECASE)
    return (m.group(1).strip() if m else str(s)).lower()

# unit word = whatever's left after stripping the leading number
def area_unit(s):
    m = re.match(r'^\s*[\d,\.]+\s*(.*)$', str(s).strip())
    return (m.group(1).strip() if m else str(s)).lower()

print('price units:')
print(df_sale['price'].apply(price_unit).value_counts())
print()
print('area units:')
print(df_sale['area'].apply(area_unit).value_counts())

price units:
price
crore    2774
lacs      189
lakh      134
arab        8
lac         3
Name: count, dtype: int64

area units:
area
marla      1672
kanal       614
sq. yd.     374
sqyd        249
sqft        199
Name: count, dtype: int64


## Convert to standard units: `price_pkr` and `area_sqft`

- price -> PKR, using Thousand / Lac(s) / Lakh / Crore / Arab as seen above
- area -> sq ft, using Marla / Kanal / Sq. Yd. / SQYD / SQFT as seen above

Conversions used (standard Pakistan real-estate values):
- 1 Marla = 272.25 sq ft
- 1 Kanal = 20 Marla = 5445 sq ft
- 1 Sq. Yd. / SQYD = 9 sq ft
- 1 SQFT = 1 sq ft

In [23]:
PRICE_UNIT_TO_PKR = {
    'thousand': 1_000,
    'lac': 100_000,
    'lacs': 100_000,
    'lakh': 100_000,
    'crore': 10_000_000,
    'arab': 1_000_000_000,
}

AREA_UNIT_TO_SQFT = {
    'marla': 272.25,
    'kanal': 5445,       # 20 Marla
    'sqft': 1,
    'sq.ft.': 1,
    'sqyd': 9,
    'sq. yd.': 9,
    'sq.yd.': 9,
}

def parse_price_pkr(text):
    """'PKR 3 Crore' / 'Rs 65 Thousand' -> 30000000.0 / 65000.0 (None if it can't be parsed)."""
    s = str(text).strip()
    m = re.match(r'^(?:PKR|Rs\.?)\s*([\d,]+(?:\.\d+)?)\s*([A-Za-z\.\s]*)$', s, re.IGNORECASE)
    if not m:
        return None
    amount = float(m.group(1).replace(',', ''))
    unit = m.group(2).strip().lower()
    multiplier = PRICE_UNIT_TO_PKR.get(unit)
    return amount * multiplier if multiplier is not None else None

def parse_area_sqft(text):
    """'10 Marla' / '2.4 Kanal' -> 2722.5 / 108900.0 (None if it can't be parsed)."""
    s = str(text).strip()
    m = re.match(r'^([\d,]+(?:\.\d+)?)\s*([A-Za-z\.\s]*)$', s)
    if not m:
        return None
    amount = float(m.group(1).replace(',', ''))
    unit = m.group(2).strip().lower()
    multiplier = AREA_UNIT_TO_SQFT.get(unit)
    return amount * multiplier if multiplier is not None else None

df_sale['price_pkr'] = df_sale['price'].apply(parse_price_pkr)
df_sale['area_sqft'] = df_sale['area'].apply(parse_area_sqft)

print('unparsed price rows:', df_sale['price_pkr'].isna().sum())
print('unparsed area rows:', df_sale['area_sqft'].isna().sum())

unparsed price rows: 0
unparsed area rows: 0


In [24]:
df_sale[['price', 'price_pkr', 'area', 'area_sqft']].head(10)

,price,price_pkr,area,area_sqft
0,PKR 3 Crore,30000000.0,10 Marla,2722.50
1,PKR 15.5 Crore,155000000.0,2.4 Kanal,13068.00
2,PKR 6.95 Crore,69500000.0,9.8 Marla,2668.05
3,PKR 14.5 Crore,145000000.0,1 Kanal,5445.00
4,PKR 14.5 Crore,145000000.0,1 Kanal,5445.00
5,PKR 3.3 Crore,33000000.0,10 Marla,2722.50
6,PKR 3.9 Crore,39000000.0,12 Marla,3267.00
7,PKR 4.65 Crore,46500000.0,13 Marla,3539.25
8,PKR 8.99 Crore,89900000.0,1 Kanal,5445.00
9,PKR 7.5 Crore,75000000.0,1 Kanal,5445.00


In [25]:
# sanity check -- if either of these is non-empty, a unit slipped through the dicts above
# and needs to be added to PRICE_UNIT_TO_PKR / AREA_UNIT_TO_SQFT
display(df_sale[df_sale['price_pkr'].isna()][['price']])
display(df_sale[df_sale['area_sqft'].isna()][['area']])

,price


,area


## Clean `bedroom` / `bath`, then add `price_per_sqft`

`bedroom` is still text -- it has `'Studio'` values plus a lot of blanks (mostly Plot/Land listings, which genuinely have no bedroom count, but some Houses/Apartments are just missing it from the scrape).

In [26]:
print('bedroom non-numeric values:')
print(df_sale[pd.to_numeric(df_sale['bedroom'], errors='coerce').isna()]['bedroom'].value_counts(dropna=False))
print()
print('rows with missing bedroom, by type:')
print(df_sale.loc[df_sale['bedroom'].isna(), 'type'].value_counts())

bedroom non-numeric values:
bedroom
NaN       519
Studio     16
Name: count, dtype: int64

rows with missing bedroom, by type:
type
Plot                162
Residential Plot    116
House                95
Other                48
Shop                 22
Commercial Plot      21
Apartment            13
Building              9
Office                7
Flat                  6
Room                  6
File                  5
Farm House            4
Villa                 2
Penthouse             1
Factory               1
Land                  1
Name: count, dtype: int64


In [27]:
# 'Studio' -> 0 bedrooms (a studio has no separate bedroom).
# Everything else that isn't a number (mostly blanks) becomes NaN -- genuinely
# unknown/not-applicable, not zero -- so it doesn't quietly bias later stats.
df_sale['bedroom'] = df_sale['bedroom'].replace('Studio', 0)
df_sale['bedroom'] = pd.to_numeric(df_sale['bedroom'], errors='coerce')

# bath is already numeric (float64) with NaNs for the same reason -- nothing to convert.
print('bedroom dtype:', df_sale['bedroom'].dtype, '| nulls:', df_sale['bedroom'].isna().sum())
print('bath dtype:', df_sale['bath'].dtype, '| nulls:', df_sale['bath'].isna().sum())

bedroom dtype: float64 | nulls: 519
bath dtype: float64 | nulls: 541


## Add `price_per_sqft`

Now that `price_pkr` and `area_sqft` are both clean numeric columns, this is just a division -- one of the rubric's own example engineered features.

In [28]:
df_sale['price_per_sqft'] = df_sale['price_pkr'] / df_sale['area_sqft']

df_sale[['price_pkr', 'area_sqft', 'price_per_sqft']].describe()

,price_pkr,area_sqft,price_per_sqft
count,3.108000e+03,3108.000000,3108.000000
mean,6.973082e+07,3401.614664,20390.775820
std,2.233383e+08,6851.365251,21059.010841
min,4.000000e+05,8.000000,133.333333
25%,1.850000e+07,1361.250000,11684.137717
50%,3.500000e+07,2160.000000,16078.971534
75%,7.000000e+07,4500.000000,21854.912764
max,9.000000e+09,217800.000000,525000.000000


In [29]:
df_sale.head()

,id,url,type,purpose,area,bedroom,bath,added,price,location,location_city,source,price_pkr,area_sqft,price_per_sqft
0,1,https://www.zameen.com/Property/askari_askari_...,Flat,For Sale,10 Marla,3.0,3.0,33 minutes ago,PKR 3 Crore,"Askari 11, Askari",Lahore,Zameen,30000000.0,2722.50,11019.283747
1,2,https://www.zameen.com/Property/gulberg_3_gulb...,Other,For Sale,2.4 Kanal,6.0,7.0,1 hour ago,PKR 15.5 Crore,"Gulberg 3 - Block M, Gulberg 3",Lahore,Zameen,155000000.0,13068.00,11861.034588
2,3,https://www.zameen.com/Property/dha_phase_5_pe...,Apartment,For Sale,9.8 Marla,3.0,4.0,3 hours ago,PKR 6.95 Crore,"Penta Square By DHA Lahore, DHA Phase 5",Lahore,Zameen,69500000.0,2668.05,26048.987088
3,4,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5.0,7.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block Y, DHA Phase 7",Lahore,Zameen,145000000.0,5445.00,26629.935721
4,5,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5.0,6.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block U, DHA Phase 7",Lahore,Zameen,145000000.0,5445.00,26629.935721
